# EMBED-mammo segmentations: load & visualize

Standalone notebook to overlay **processed mammography segmentations** on DICOM images for patients in an annotations table.

## What you need

Point the configuration cell at three inputs (any folder layout is fine):

| Input | Description |
|-------|-------------|
| **Annotations CSV** | One row per image. Required columns: `acc_anon` (patient ID), `laterality`, `view_position`, `anon_dicom_path`. |
| **Segmentations root** | Folder containing one subfolder per patient: `<acc_anon>/{VIEW}_Segmentations.seg.nrrd` (e.g. `LCC_Segmentations.seg.nrrd`). |
| **DICOM images** | Paths in `anon_dicom_path`, or set `IMAGE_ROOT` to rebase cohort-relative CSV paths onto your local DICOM tree. |

Each `.seg.nrrd` is a Slicer-style labelmap. Segment names are in the NRRD header (`Segment{i}_Name`), typically `{VIEW}-{lesion}` (e.g. `LCC-Mass1`, `LMLO-MainCalcifications`).

## Workflow

1. **Setup** - set `CSV_PATH`, `SEGMENTATIONS_ROOT`, `IMAGE_ROOT`, and optionally `PATIENT_ID`.
2. **Browse** - list patients with both an image and a segmentation on at least one view.
3. **Visualize** - overlay segment masks on the matching DICOM views.

## Dependencies

```bash
pip install pydicom pynrrd matplotlib pylibjpeg pylibjpeg-libjpeg
# optional interactive patient picker:
pip install ipywidgets
# optional external interactive plot windows:
pip install pyqt6
```


In [ ]:
# Imports
from __future__ import annotations

import ast
import csv
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple, Union

import numpy as np

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    import pydicom
except ImportError:
    pydicom = None

try:
    import nrrd
except ImportError:
    nrrd = None

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None

    def display(obj: object) -> None:
        print(obj)



In [ ]:
# Paths & configuration - edit for your environment
CSV_PATH = Path("/path/to/df_first_annotations.csv")
SEGMENTATIONS_ROOT = Path("/path/to/segmentations")  # <acc_anon>/{VIEW}_Segmentations.seg.nrrd

# Rebases cohort-relative paths from the CSV, or None to use anon_dicom_path as-is.
IMAGE_ROOT: Optional[Union[Path, str]] = Path("/path/to/dicom/root")

VIEWS = ("LCC", "LMLO", "RCC", "RMLO")
PATIENT_ID: Optional[str] = None  # e.g. "1234567890123456"; fallback when ipywidgets is absent

OVERLAY_ALPHA = 0.75
USE_CSV_COLOR_SCHEME = True

# Visualization layout: "grid" (2x2) or "separate" (one figure per view)
PLOT_LAYOUT: str = "grid"

# Plot output: "inline" (notebook) or "window" (interactive Qt window)
PLOT_DISPLAY: str = "window"

print(f"Annotations CSV:    {CSV_PATH} ({'ok' if CSV_PATH.exists() else 'missing'})")
print(f"Segmentations root: {SEGMENTATIONS_ROOT} ({'ok' if SEGMENTATIONS_ROOT.exists() else 'missing'})")
print(f"Image root:         {IMAGE_ROOT}")


In [ ]:
# Path resolution & records
COHORT_PATH_RE = re.compile(r"(cohort_\d+/.*)")
EXPECTED_VIEWS = VIEWS

@dataclass(frozen=True)
class ViewRecord:
    """DICOM and segmentation paths for one patient view."""
    patient_id: str
    view: str
    dicom_path: Optional[Path]
    seg_path: Optional[Path]


def view_code_from_filename(path: Path) -> str:
    """Return view code from a filename like LCC_Segmentations.seg.nrrd."""
    stem = path.name.replace(".seg.nrrd", "").replace(".nrrd", "")
    if not stem.endswith("_Segmentations"):
        raise ValueError(f"Unexpected segmentation filename: {path.name}")

    return stem.split("_Segmentations")[0]


def laterality_view_to_code(laterality: str, view_position: str) -> str:
    """Map CSV laterality + view_position to LCC / RCC / LMLO / RMLO."""
    prefix = "L" if str(laterality).strip().upper() == "L" else "R"

    return f"{prefix}{view_position.strip().upper()}"


def _read_csv_rows(csv_path: Path) -> List[Dict[str, str]]:
    with csv_path.open("r", newline="") as handle:
        return list(csv.DictReader(handle))


def _resolve_dicom_path(anon_dicom_path: str, image_root: Optional[Union[Path, str]]) -> Path:
    if image_root is None:
        return Path(anon_dicom_path)

    match = COHORT_PATH_RE.search(anon_dicom_path)
    if match is None:
        raise ValueError(f"Could not extract cohort-relative path from: {anon_dicom_path}")

    return Path(image_root) / match.group(1)


def list_patients_with_segmentations(seg_root: Path) -> List[str]:
    """Return sorted acc_anon IDs that have a segmentation folder."""
    if not seg_root.exists():
        return []

    return sorted(
        path.name
        for path in seg_root.iterdir()
        if path.is_dir() and any(path.glob("*_Segmentations.seg.nrrd"))
    )


def summarize_patient_segmentations(seg_root: Path, patient_id: str) -> Dict[str, object]:
    """Summarize segmentation files available for one patient."""
    patient_dir = seg_root / patient_id
    seg_paths = sorted(patient_dir.glob("*_Segmentations.seg.nrrd")) if patient_dir.exists() else []
    views = [view_code_from_filename(path) for path in seg_paths]

    return {
        "patient_id": patient_id,
        "n_views": len(views),
        "views": views,
        "missing_views": [view for view in EXPECTED_VIEWS if view not in views],
        "seg_paths": {view: path for view, path in zip(views, seg_paths)},
    }


def build_dicom_paths_by_view(
    csv_path: Path,
    patient_id: str,
    image_root: Optional[Union[Path, str]],
) -> Dict[str, Path]:
    view_to_path: Dict[str, Path] = {}

    for row in _read_csv_rows(csv_path):
        if row.get("acc_anon") != patient_id:
            continue

        laterality = row.get("laterality")
        view_position = row.get("view_position")
        anon_path = row.get("anon_dicom_path")
        if not laterality or not view_position or not anon_path:
            continue

        view = laterality_view_to_code(laterality, view_position)
        view_to_path[view] = _resolve_dicom_path(anon_path, image_root)

    return view_to_path


def build_processed_seg_paths_by_view(seg_root: Path, patient_id: str) -> Dict[str, Path]:
    patient_dir = seg_root / patient_id
    if not patient_dir.exists():
        return {}

    view_to_path: Dict[str, Path] = {}
    for path in sorted(patient_dir.glob("*_Segmentations.seg.nrrd")):
        view_to_path[view_code_from_filename(path)] = path

    return view_to_path


def build_view_records(
    patient_id: str,
    views_to_show: Sequence[str],
    csv_path: Path,
    seg_root: Path,
    image_root: Optional[Union[Path, str]],
) -> List[ViewRecord]:
    dcm_by_view = build_dicom_paths_by_view(csv_path, patient_id, image_root)
    seg_by_view = build_processed_seg_paths_by_view(seg_root, patient_id)

    return [
        ViewRecord(
            patient_id=patient_id,
            view=view,
            dicom_path=dcm_by_view.get(view),
            seg_path=seg_by_view.get(view),
        )
        for view in views_to_show
    ]


def summarize_patient_availability(
    patient_id: str,
    csv_path: Path,
    seg_root: Path,
    image_root: Optional[Union[Path, str]],
) -> Dict[str, object]:
    """Summarize which views have accessible DICOMs and segmentations.

    Parameters
    ----------
    patient_id
        `acc_anon` identifier.
    csv_path
        Path to `df_first_annotations.csv`.
    seg_root
        Processed segmentations root.
    image_root
        Optional base directory for resolving DICOM paths from the CSV.

    Returns
    -------
    dict[str, object]
        Keys: `patient_id`, `dicom_views`, `seg_views`, `both_views`.
    """
    dicom_by_view = build_dicom_paths_by_view(csv_path, patient_id, image_root)
    seg_by_view = build_processed_seg_paths_by_view(seg_root, patient_id)

    dicom_views = sorted(view for view, path in dicom_by_view.items() if path.exists())
    seg_views = sorted(seg_by_view.keys())
    both_views = sorted(set(dicom_views) & set(seg_views))

    return {
        "patient_id": patient_id,
        "dicom_views": dicom_views,
        "seg_views": seg_views,
        "both_views": both_views,
    }


def list_patients_with_dicom_and_segmentations(
    csv_path: Path,
    seg_root: Path,
    image_root: Optional[Union[Path, str]],
) -> List[str]:
    """Return patients with at least one view that has both DICOM and segmentation."""
    return [
        patient_id
        for patient_id in list_patients_with_segmentations(seg_root)
        if summarize_patient_availability(patient_id, csv_path, seg_root, image_root)["both_views"]
    ]


def format_view_status(record: ViewRecord) -> str:
    """One-line availability summary for a view."""
    if record.dicom_path is None:
        dicom_status = "dicom=MISSING"
    elif record.dicom_path.exists():
        dicom_status = "dicom=OK"
    else:
        dicom_status = "dicom=NOT_FOUND"

    seg_status = "seg=OK" if record.seg_path is not None else "seg=MISSING"

    return f"{record.view}: {dicom_status} | {seg_status}"


In [ ]:
# Loading DICOMs and .seg.nrrd files

def load_dicom_pixel_array(path: Path) -> np.ndarray:
    """Load a DICOM file into a 2D numpy array."""
    if pydicom is None:
        raise RuntimeError("Install pydicom: pip install pydicom")

    ds = pydicom.dcmread(str(path))

    try:
        pixel_array = ds.pixel_array
    except RuntimeError as exc:
        message = str(exc)
        if "Unable to decompress" in message or "plugins are missing dependencies" in message:
            raise RuntimeError(
                "Compressed DICOM ? install a decoder: "
                "pip install pylibjpeg pylibjpeg-libjpeg"
            ) from exc

        raise

    return np.asarray(pixel_array)


def load_seg_nrrd(path: Path) -> Tuple[np.ndarray, Dict[int, str]]:
    """Load a `.seg.nrrd` and return (labelmap array, segment index -> name)."""
    if nrrd is None:
        raise RuntimeError("Install pynrrd: pip install pynrrd")

    data, header = nrrd.read(str(path))

    segment_names: Dict[int, str] = {}
    for key, value in header.items():
        if not isinstance(key, str):
            continue

        match = re.fullmatch(r"Segment(\d+)_Name", key)
        if match is not None:
            segment_names[int(match.group(1))] = str(value)

    return np.asarray(data), segment_names


def squeeze_mammo_image(array: np.ndarray) -> np.ndarray:
    squeezed = np.squeeze(array)
    if squeezed.ndim != 2:
        raise ValueError(f"Expected 2D after squeeze, got {array.shape} -> {squeezed.shape}")

    return squeezed


def seg_to_binary_masks(seg_data: np.ndarray) -> List[np.ndarray]:
    """Convert preprocessed NRRD data to a list of 2D boolean masks."""
    data = np.asarray(seg_data)

    if data.ndim == 2:
        return [data.astype(bool)]

    if data.ndim == 4 and data.shape[-1] == 1:
        return [squeeze_mammo_image(data[i]) > 0 for i in range(data.shape[0])]

    if data.ndim == 3 and data.shape[0] <= 32:
        return [squeeze_mammo_image(data[i]) > 0 for i in range(data.shape[0])]

    raise ValueError(f"Unsupported seg shape {data.shape}; expected (n_segments, H, W, 1).")


def load_seg_masks(path: Path) -> Tuple[List[np.ndarray], Dict[int, str]]:
    """Load a `.seg.nrrd` and return (masks, segment_names).

    Parameters
    ----------
    path
        Path to the `.seg.nrrd` file.

    Returns
    -------
    tuple[list[numpy.ndarray], dict[int, str]]
        `(masks, segment_names)` where `masks` is a list of 2D boolean arrays.
    """
    seg_data, segment_names = load_seg_nrrd(path)

    masks = seg_to_binary_masks(seg_data)

    return masks, segment_names


In [ ]:
# Overlay colors (consistent across patients)

def parse_csv_json_list(raw_value: Optional[str]) -> List[str]:
    if raw_value is None:
        return []

    text = str(raw_value).strip()
    if not text or text.lower() == "nan" or text == "[]":
        return []

    try:
        return [str(x) for x in json.loads(text)]
    except Exception:
        pass

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except Exception:
        pass

    return []


def lesion_suffix_from_label(label: str) -> str:
    if "-" not in label:
        return label

    return label.split("-", 1)[1]


def build_possible_lesion_suffixes_from_csv(csv_path: Path) -> List[str]:
    rows = _read_csv_rows(csv_path)
    max_masses = 0
    suffixes: set[str] = set()

    for row in rows:
        try:
            max_masses = max(max_masses, int(float(row.get("num_masses") or 0)))
        except Exception:
            pass

        if str(row.get("asymmetry") or "").strip().lower() == "yes":
            suffixes.add("Asymmetry")

        if str(row.get("architectural_distortion") or "").strip().lower() == "yes":
            suffixes.add("MainDistortion")

        if str(row.get("calcifications") or "").strip() not in ("", "0", "No", "no", "None", "nan"):
            suffixes.add("MainCalcifications")

        for feature in parse_csv_json_list(row.get("associated_features")):
            suffixes.add(feature)

    for i in range(1, max_masses + 1):
        suffixes.add(f"Mass{i}")

    ordered: List[str] = [f"Mass{i}" for i in range(1, max_masses + 1)]
    for core in ("Asymmetry", "MainDistortion", "MainCalcifications"):
        if core in suffixes:
            ordered.append(core)

    ordered.extend(sorted(s for s in suffixes if s not in set(ordered)))

    return ordered


def build_color_by_lesion_suffix(csv_path: Path) -> Dict[str, Tuple[float, float, float]]:
    suffixes = build_possible_lesion_suffixes_from_csv(csv_path)

    if plt is not None:
        cmap = plt.colormaps.get_cmap("tab20")
        colors = [cmap(i % 20)[:3] for i in range(len(suffixes))]
    else:
        colors = [(i / max(1, len(suffixes)), 0.7, 0.9) for i in range(len(suffixes))]

    return {suffix: (float(c[0]), float(c[1]), float(c[2])) for suffix, c in zip(suffixes, colors)}


In [ ]:
# Visualization

try:
    from IPython import get_ipython
except ImportError:
    def get_ipython() -> None:
        return None


def _ensure_matplotlib() -> None:
    if plt is None:
        raise RuntimeError("Install matplotlib: pip install matplotlib")


def _configure_matplotlib_display(display_mode: Optional[str] = None) -> str:
    """Switch between inline notebook output and an interactive Qt window."""
    mode = display_mode or PLOT_DISPLAY
    if mode not in ("inline", "window"):
        raise ValueError(f"Unknown PLOT_DISPLAY={mode!r}. Use 'inline' or 'window'.")

    ip = get_ipython()
    if ip is not None:
        if mode == "window":
            ip.run_line_magic("matplotlib", "qt")
        else:
            ip.run_line_magic("matplotlib", "inline")

    return mode


def _show_figure() -> None:
    """Display the active figure using the configured output mode."""
    plt.show(block=False)


def normalize_to_unit_interval(image: np.ndarray) -> np.ndarray:
    img = image.astype(np.float32)
    finite = np.isfinite(img)
    if not np.any(finite):
        return np.zeros_like(img, dtype=np.float32)

    vmin = float(np.percentile(img[finite], 1))
    vmax = float(np.percentile(img[finite], 99))
    if vmax <= vmin:
        return np.zeros_like(img, dtype=np.float32)

    return np.clip((img - vmin) / (vmax - vmin), 0.0, 1.0)


def align_mask_to_image(mask_2d: np.ndarray, image_2d: np.ndarray) -> np.ndarray:
    if mask_2d.shape == image_2d.shape:
        return mask_2d

    if mask_2d.T.shape == image_2d.shape:
        return mask_2d.T

    raise ValueError(f"Mask {mask_2d.shape} cannot align to image {image_2d.shape}")


def _corner_slices(height: int, width: int, frac: float = 0.2) -> Dict[str, Tuple[slice, slice]]:
    """Return row/col slices for each corner patch."""
    dh = max(1, int(round(frac * height)))
    dw = max(1, int(round(frac * width)))

    return {
        "upper left": (slice(0, dh), slice(0, dw)),
        "upper right": (slice(0, dh), slice(width - dw, width)),
        "lower left": (slice(height - dh, height), slice(0, dw)),
        "lower right": (slice(height - dh, height), slice(width - dw, width)),
    }


def _build_tissue_proxy(
    base_image: np.ndarray,
    masks: Sequence[np.ndarray],
) -> np.ndarray:
    """Estimate breast/segmentation occupancy to avoid placing the legend on tissue."""
    tissue = base_image > 0.08

    for mask in masks:
        if mask is None:
            continue

        tissue |= align_mask_to_image(mask.astype(bool), base_image)

    return tissue


def _choose_legend_loc(
    base_image: np.ndarray,
    masks: Sequence[np.ndarray],
    view: Optional[str] = None,
) -> str:
    """Pick the corner with the least tissue overlap."""
    height, width = base_image.shape
    tissue = _build_tissue_proxy(base_image, masks)

    preferred_order = {
        "LCC": ("upper right", "lower right", "upper left", "lower left"),
        "RCC": ("upper left", "lower left", "upper right", "lower right"),
        "LMLO": ("lower right", "upper right", "lower left", "upper left"),
        "RMLO": ("lower left", "upper left", "lower right", "upper right"),
    }

    scores: Dict[str, float] = {}
    for loc, (row_slice, col_slice) in _corner_slices(height, width).items():
        scores[loc] = float(np.mean(tissue[row_slice, col_slice]))

    min_score = min(scores.values())
    candidates = [loc for loc, score in scores.items() if score <= min_score + 0.02]

    if view in preferred_order:
        for loc in preferred_order[view]:
            if loc in candidates:
                return loc

    return min(scores, key=scores.get)


def plot_view_with_overlays(
    ax: object,
    image_2d: np.ndarray,
    masks: Sequence[np.ndarray],
    names: Optional[Dict[int, str]],
    alpha: float,
    title: str,
    color_by_suffix: Optional[Dict[str, Tuple[float, float, float]]] = None,
) -> None:
    _ensure_matplotlib()
    from matplotlib.patches import Patch

    base = normalize_to_unit_interval(image_2d)
    ax.imshow(base, cmap="gray", interpolation="nearest")

    legend_labels: List[str] = []
    legend_colors: List[Tuple[float, float, float]] = []
    rng = np.random.default_rng(0)

    for idx, mask in enumerate(masks):
        aligned = align_mask_to_image(mask.astype(bool), base)
        if not np.any(aligned):
            continue

        label = names.get(idx) if names else None
        if label is None:
            label = f"seg_{idx}"

        suffix = lesion_suffix_from_label(str(label))
        if color_by_suffix and suffix in color_by_suffix:
            color = color_by_suffix[suffix]
        else:
            color = tuple(rng.random(3))

        overlay = np.zeros((*base.shape, 4), dtype=np.float32)
        overlay[..., :3] = color
        overlay[..., 3] = alpha * aligned.astype(np.float32)
        ax.imshow(overlay)

        legend_labels.append(str(label))
        legend_colors.append(color)

    if legend_labels:
        handles = [Patch(facecolor=c, edgecolor="none", label=lbl) for lbl, c in zip(legend_labels, legend_colors)]
        legend_loc = _choose_legend_loc(base, masks, view=title)
        ax.legend(
            handles=handles,
            loc=legend_loc,
            fontsize=8,
            framealpha=0.9,
            facecolor="white",
            edgecolor="none",
        )

    ax.set_title(title, fontsize=12, fontweight="semibold")
    ax.axis("off")


def overlay_masks_inline(
    image_2d: np.ndarray,
    masks: Sequence[np.ndarray],
    names: Optional[Dict[int, str]] = None,
    alpha: float = OVERLAY_ALPHA,
    title: Optional[str] = None,
    color_by_suffix: Optional[Dict[str, Tuple[float, float, float]]] = None,
) -> None:
    """Show one view with colored mask overlays in its own figure."""
    _ensure_matplotlib()

    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    plot_view_with_overlays(
        ax=ax,
        image_2d=image_2d,
        masks=masks,
        names=names,
        alpha=alpha,
        title=title or "",
        color_by_suffix=color_by_suffix,
    )
    _show_figure()


def load_patient_view_data(
    patient_id: str,
    views_to_show: Sequence[str] = VIEWS,
) -> List[Tuple[ViewRecord, np.ndarray, List[np.ndarray], Dict[int, str]]]:
    """Load DICOM images and segmentation masks for all available views."""
    records = build_view_records(
        patient_id=patient_id,
        views_to_show=views_to_show,
        csv_path=CSV_PATH,
        seg_root=SEGMENTATIONS_ROOT,
        image_root=IMAGE_ROOT,
    )

    payloads: List[Tuple[ViewRecord, np.ndarray, List[np.ndarray], Dict[int, str]]] = []

    for rec in records:
        if rec.dicom_path is None or not rec.dicom_path.exists():
            continue

        image_2d = load_dicom_pixel_array(rec.dicom_path)
        masks: List[np.ndarray] = []
        names: Dict[int, str] = {}

        if rec.seg_path is not None:
            seg_data, seg_names = load_seg_nrrd(rec.seg_path)
            masks = seg_to_binary_masks(seg_data)
            names = seg_names

        payloads.append((rec, image_2d, masks, names))

    return payloads


def visualize_patient(
    patient_id: str,
    views_to_show: Sequence[str] = VIEWS,
    show_status: bool = True,
    plot_layout: Optional[str] = None,
    plot_display: Optional[str] = None,
) -> None:
    """Show mammography views with segmentation overlays.

    Parameters
    ----------
    patient_id
        `acc_anon` identifier.
    views_to_show
        View codes to attempt.
    show_status
        Print per-view DICOM/segmentation availability.
    plot_layout
        `"grid"` for a single 2x2 figure, `"separate"` for one figure per view.
        Defaults to `PLOT_LAYOUT` from the configuration cell.
    plot_display
        `"inline"` for notebook output, `"window"` for an interactive Qt window.
        Defaults to `PLOT_DISPLAY` from the configuration cell.
    """
    _ensure_matplotlib()
    _configure_matplotlib_display(plot_display)

    layout = plot_layout or PLOT_LAYOUT
    if layout not in ("grid", "separate"):
        raise ValueError(f"Unknown plot_layout={layout!r}. Use 'grid' or 'separate'.")

    records = build_view_records(
        patient_id=patient_id,
        views_to_show=views_to_show,
        csv_path=CSV_PATH,
        seg_root=SEGMENTATIONS_ROOT,
        image_root=IMAGE_ROOT,
    )

    if show_status:
        print(f"acc_anon={patient_id}")
        for rec in records:
            print(f"  {format_view_status(rec)}")

    color_by_suffix = build_color_by_lesion_suffix(CSV_PATH) if USE_CSV_COLOR_SCHEME else None
    payloads = load_patient_view_data(patient_id, views_to_show)

    if layout == "separate":
        for rec, image_2d, masks, names in payloads:
            title = f"{rec.patient_id} | {rec.view}"
            overlay_masks_inline(
                image_2d=image_2d,
                masks=masks,
                names=names,
                alpha=OVERLAY_ALPHA,
                title=title,
                color_by_suffix=color_by_suffix,
            )

        return

    view_to_payload = {rec.view: (rec, img, masks, names) for rec, img, masks, names in payloads}

    ncols = 2
    nrows = int(np.ceil(len(views_to_show) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows), constrained_layout=True)
    fig.suptitle(f"acc_anon={patient_id}", fontsize=14, fontweight="bold")

    for idx, ax in enumerate(np.atleast_1d(axes).ravel()):
        if idx >= len(views_to_show):
            ax.axis("off")
            continue

        view = views_to_show[idx]
        payload = view_to_payload.get(view)
        if payload is None:
            ax.set_title(f"{view} (unavailable)", fontsize=12)
            ax.axis("off")
            continue

        rec, image_2d, masks, names = payload
        plot_view_with_overlays(
            ax=ax,
            image_2d=image_2d,
            masks=masks,
            names=names,
            alpha=OVERLAY_ALPHA,
            title=rec.view,
            color_by_suffix=color_by_suffix,
        )

    _show_figure()


## Browse patients

Run the cell below to list patients with **at least one view** where both a DICOM image (on disk) and a processed segmentation are available. Each line shows which image views are accessible for that patient.

Use the dropdown in the next section to visualize any listed patient.

In [ ]:
# Dataset overview
_all_seg_patients = list_patients_with_segmentations(SEGMENTATIONS_ROOT)
PATIENT_IDS = list_patients_with_dicom_and_segmentations(
    CSV_PATH,
    SEGMENTATIONS_ROOT,
    IMAGE_ROOT,
)

print(
    f"{len(PATIENT_IDS)} patients with image + segmentation "
    f"(of {len(_all_seg_patients)} with segmentations)"
)

if not PATIENT_IDS and _all_seg_patients:
    print("\nNo accessible DICOMs ? set IMAGE_ROOT in the setup cell.")
else:
    for patient_id in PATIENT_IDS:
        summary = summarize_patient_availability(
            patient_id,
            CSV_PATH,
            SEGMENTATIONS_ROOT,
            IMAGE_ROOT,
        )
        views = ", ".join(summary["dicom_views"]) or "?"

        print(f"{patient_id}  {views}")


## Visualize

With **ipywidgets** installed, use the dropdown. Otherwise set `PATIENT_ID` in the configuration cell and run `visualize_patient(PATIENT_ID)`.

Set `PLOT_LAYOUT = "grid"` for a single 2x2 figure, or `"separate"` for one figure per view.

Set `PLOT_DISPLAY = "inline"` for notebook output, or `"window"` for an interactive Qt window (`pip install pyqt6`).

Status codes: `dicom=MISSING` (no CSV row), `dicom=NOT_FOUND` (path not on disk - set `IMAGE_ROOT`), `seg=MISSING` (no `.seg.nrrd` for that view).


In [ ]:
# Interactive patient viewer
if widgets is not None and PATIENT_IDS:
    default_patient = PATIENT_ID if PATIENT_ID in PATIENT_IDS else PATIENT_IDS[0]
    picker = widgets.Dropdown(
        options=PATIENT_IDS,
        value=default_patient,
        description="acc_anon:",
        layout=widgets.Layout(width="420px"),
    )
    out = widgets.interactive_output(
        visualize_patient,
        {"patient_id": picker, "show_status": widgets.fixed(True)},
    )
    display(picker, out)
elif PATIENT_IDS:
    visualize_patient(PATIENT_ID or PATIENT_IDS[0])
else:
    print("No patients found. Edit CSV_PATH, SEGMENTATIONS_ROOT, and IMAGE_ROOT in the configuration cell.")


## Minimal loading example

To use segmentations in your own code (without plotting):

```python
patient_id = "<acc_anon>"
view = "LMLO"

seg_path = SEGMENTATIONS_ROOT / patient_id / f"{view}_Segmentations.seg.nrrd"
masks, segment_names = load_seg_masks(seg_path)

for idx, mask in enumerate(masks):
    name = segment_names.get(idx, f"seg_{idx}")

    print(name, mask.shape, int(mask.sum()))
```